# VINCULUM TERRAIN OPTIMIZER
Analyzes optimal district placement, contour efficiency, network topology, and chroma contrast.

**Run on Kaggle GPU** — computes 50K elevation samples, overlap detection, desire path prediction.

**Output:** Optimized SEED JSON for VINCULUM_TERRAIN.html


In [1]:
import numpy as np, json, math, copy
from collections import defaultdict
print("Vinculum Terrain Optimizer — Kaggle GPU")
print(f"NumPy: {np.__version__}")


In [2]:
class TerrainNoise:
    def noise(self, x, y):
        nx = int(np.floor(x / 50))
        ny = int(np.floor(y / 50))
        fx = (x % 50) / 50.0
        fy = (y % 50) / 50.0
        r1 = np.sin(nx * 12.9898 + ny * 78.233) * 43758.5453
        r2 = np.sin((nx + 1) * 12.9898 + ny * 78.233) * 43758.5453
        r3 = np.sin(nx * 12.9898 + (ny + 1) * 78.233) * 43758.5453
        r4 = np.sin((nx + 1) * 12.9898 + (ny + 1) * 78.233) * 43758.5453
        n1 = r1 - np.floor(r1)
        n2 = r2 - np.floor(r2)
        n3 = r3 - np.floor(r3)
        n4 = r4 - np.floor(r4)
        ix1 = n1 + (n2 - n1) * fx
        ix2 = n3 + (n4 - n3) * fx
        return ix1 + (ix2 - ix1) * fy

    def get_elevation(self, x, y, d):
        base = self.noise(x, y) * 35
        if d:
            dx = x - d["x"]
            dy = y - d["y"]
            f = 1 - min(1.0, math.sqrt(dx*dx+dy*dy) / d["r"])
            base += d["height"] * f * f
            if d["id"] == "CEN": base += 50 * f
        if base < 20: base = 17
        return base

tn = TerrainNoise()
print("Noise generator ready")


In [3]:
SEED = {"districts":[
  {"id":"FIN","x":0,"y":-280,"r":260,"hue":190,"height":90,"density":0.9},
  {"id":"HAR","x":420,"y":120,"r":300,"hue":45,"height":25,"density":0.4},
  {"id":"ECO","x":-380,"y":40,"r":220,"hue":100,"height":50,"density":0.3},
  {"id":"RES","x":180,"y":280,"r":240,"hue":55,"height":65,"density":0.7},
  {"id":"IND","x":-220,"y":380,"r":280,"hue":320,"height":40,"density":0.6},
  {"id":"RND","x":-140,"y":-140,"r":180,"hue":150,"height":60,"density":0.5},
  {"id":"CEN","x":0,"y":0,"r":100,"hue":0,"height":110,"density":1.0}]}
print("7 districts loaded")


In [4]:
# 1. ELEVATION ANALYSIS (50K samples)
delev = defaultdict(list)
S = 50000
for _ in range(S):
    x = np.random.uniform(-600,600)
    y = np.random.uniform(-400,400)
    best = None; bd = float("inf")
    for d in SEED["districts"]:
        dist = math.sqrt((x-d["x"])**2 + (y-d["y"])**2)
        if dist < d["r"] and dist < bd:
            best = d; bd = dist
    e = tn.get_elevation(x, y, best)
    delev[best["id"] if best else "WILD"].append(e)
print("ELEVATION DISTRIBUTION (50K samples):")
print(f"  {'District':8s} {'Count':>6s} {'Mean':>7s} {'Max':>7s}")
for k in sorted(delev.keys()):
    v = np.array(delev[k])
    print(f"  {k:8s} {len(v):6,} {v.mean():7.1f} {v.max():7.1f}")
all_elev = np.array([e for v in delev.values() for e in v])
print(f"\n  Water area (<20): {np.sum(all_elev<20)/S*100:.1f}%")
print(f"  Urban area (>40): {np.sum(all_elev>40)/S*100:.1f}%")


In [5]:
# 2. OVERLAP DETECTION
overlaps = []
for i,a in enumerate(SEED["districts"]):
    for j,b in enumerate(SEED["districts"]):
        if i >= j: continue
        d = math.sqrt((a["x"]-b["x"])**2 + (a["y"]-b["y"])**2)
        ov = a["r"] + b["r"] - d
        if ov > 0:
            overlaps.append((a["id"], b["id"], ov, ov/min(a["r"],b["r"])))
overlaps.sort(key=lambda x: -x[2])
print("DISTRICT OVERLAPS:" if overlaps else "NO OVERLAPS — all districts cleanly separated")
for a,b,ov,ratio in overlaps:
    bar = chr(9608)*int(ratio*20) + chr(9617)*(20-int(ratio*20))
    print(f"  {a}-{b}: {ov:6.0f} units  {bar}")


In [6]:
# 3. MAGLEV NETWORK TOPOLOGY
MAGLEV = [("FIN","CEN",10),("CEN","HAR",8),("CEN","ECO",8),("CEN","RES",10),
         ("RES","IND",6),("RND","FIN",7),("RND","ECO",6),("IND","HAR",5)]
adj = defaultdict(set)
for a,b,w in MAGLEV:
    adj[a].add(b); adj[b].add(a)
n = len(SEED["districts"])
print("NETWORK TOPOLOGY:")
for d in SEED["districts"]:
    c = len(adj[d["id"]])
    bar = chr(9608)*c + chr(9617)*((n-1)-c)
    print(f"  {d["id"]}: {c}/{n-1} {bar}")
max_edges = n*(n-1)//2
print(f"\n  Density: {len(MAGLEV)}/{max_edges} = {len(MAGLEV)/max_edges:.2f}")
print(f"  Total edges: {len(MAGLEV)}")


In [7]:
# 4. CHROMA CONTRAST ANALYSIS
def hsl_to_rgb(h):
    s, l_ = 0.70, 0.45
    c = (1 - abs(2*l_ - 1)) * s
    x = c * (1 - abs((h / 60) % 2 - 1))
    m = l_ - c/2
    r,g,b = [(c,x,0),(x,c,0),(0,c,x),(0,x,c),(x,0,c),(c,0,x)][int(h//60)%6]
    return np.array([r+m, g+m, b+m])

pairs = []
for a in SEED["districts"]:
    for b in SEED["districts"]:
        if a["id"] >= b["id"]: continue
        de = np.sqrt(np.sum((hsl_to_rgb(a["hue"]) - hsl_to_rgb(b["hue"]))**2))
        pairs.append((a["id"], b["id"], de))
pairs.sort(key=lambda x: x[2])
print('CHROMA CONTRAST:')
print('  Worst (colors too similar):')
for a,b,de in pairs[:2]:
    print(f'    {a}-{b}  deltaE={de:.3f}  (fix these first)')
print('  Best contrast:')
    print(f'    {a}-{b}  deltaE={de:.3f}')

hues = [d["hue"] for d in SEED["districts"]]
print(f'\n  Hue range: {min(hues)}-{max(hues)} deg (spread={max(hues)-min(hues)}deg)')
print(f'  Ideal gap: {360//7}deg per district')


In [8]:
# 5. DESIRE PATH ANALYSIS
paths = []
for a in SEED["districts"]:
    for b in SEED["districts"]:
        if a["id"] >= b["id"]: continue
        cost = 0
        for t in range(20):
            f = t / 19
            px = a["x"] + (b["x"] - a["x"]) * f
            py = a["y"] + (b["y"] - a["y"]) * f
            best = None; bd = float("inf")
            for d in SEED["districts"]:
                dist = math.sqrt((px-d["x"])**2 + (py-d["y"])**2)
                if dist < d["r"] and dist < bd: best = d; bd = dist
            cost += max(1, tn.get_elevation(px, py, best) - 15) * 30
        paths.append((a["id"], b["id"], cost))
paths.sort(key=lambda x: x[2])
print('DESIRE PATHS — easiest routes:')
for a,b,cost in paths[:4]:
    print(f'  {a}-{b}  cost={cost:.0f}')
print()
print('Hardest routes:')
for a,b,cost in paths[-4:]:
    print(f'  {a}-{b}  cost={cost:.0f}')


In [9]:
# 6. RECOMMENDATIONS
print('RECOMMENDATIONS:')

# Maglev gaps
existing = {tuple(sorted([a,b])) for a,b,w in MAGLEV}
gaps = []
for a in SEED["districts"]:
    for b in SEED["districts"]:
        p = tuple(sorted([a['id'], b['id']]))
        if a['id'] < b['id'] and p not in existing:
            d = math.sqrt((a["x"]-b["x"])**2 + (a["y"]-b["y"])**2)
            gaps.append((p, d))
gaps.sort(key=lambda x: x[1])
print('Maglev connections to add:')
for (a,b),d in gaps[:3]:
    print(f'  {a}-{b}  (distance={d:.0f} units)')

# Chroma fix
wp = pairs[0]
a = next(d for d in SEED["districts"] if d["id"]==wp[0])
b = next(d for d in SEED["districts"] if d["id"]==wp[1])
diff = abs(a["hue"] - b["hue"])
if diff < 40:
    print(f"\nChroma fix: shift {wp[1]} hue from {b["hue"]} to {(b["hue"]+60)%360} deg")


In [10]:
# 7. OPTIMIZATION (hill-climb for zero overlap)
best_score = -1e9
best_seed = copy.deepcopy(SEED)
for it in range(300):
    cand = copy.deepcopy(SEED)
    for d in cand["districts"]:
        d["x"] += np.random.randint(-20, 21)
        d["y"] += np.random.randint(-20, 21)
    score = 0
    for i,a in enumerate(cand["districts"]):
        for j,b in enumerate(cand["districts"]):
            if i >= j: continue
            d = math.sqrt((a["x"]-b["x"])**2 + (a["y"]-b["y"])**2)
            ov = a["r"] + b["r"] - d
            if ov > 0: score -= ov * ov  # squared penalty
    if score > best_score:
        best_score = score
        best_seed = copy.deepcopy(cand)

print('OPTIMIZED DISTRICT POSITIONS:')
for d in best_seed["districts"]:
    o = next(x for x in SEED["districts"] if x["id"]==d["id"])
    dx = d["x"] - o["x"]
    dy = d["y"] - o["y"]
    print(f"  {d["id"]}: ({d["x"]:+4d}, {d["y"]:+4d})  shift=({dx:+3d}, {dy:+3d})")

# Verify no overlaps in optimized
ov2 = []
for i,a in enumerate(best_seed["districts"]):
    for j,b in enumerate(best_seed["districts"]):
        if i >= j: continue
        d = math.sqrt((a["x"]-b["x"])**2 + (a["y"]-b["y"])**2)
        if a["r"] + b["r"] > d: ov2.append((a["id"],b["id"]))
print(f"\nOverlaps after optimization: {len(ov2)}" if ov2 else "\nZERO OVERLAPS after optimization")


In [11]:
# 8. EXPORT
output = json.dumps(best_seed, indent=2)
with open("/kaggle/working/vinculum_optimized_seed.json", "w") as f:
    f.write(output)
print('OPTIMIZED SEED (paste into VINCULUM_TERRAIN.html):')
print(output)
print(f'\nSaved: /kaggle/working/vinculum_optimized_seed.json ({len(output)} chars)')
